In [ ]:
!pip install flask transformers requests

In [ ]:
%%writefile flask_chatbot_app.py
from flask import Flask, request, jsonify
from transformers import pipeline, TFAutoModelForSeq2SeqLM, AutoTokenizer

app = Flask(__name__)

# Load the Hugging Face model

model_name = 'google/flan-t5-xl'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)
sentiment_model = pipeline("sentiment-analysis")

@app.route('/chatbot', methods=['POST'])
def chatbot():
    user_input = request.json.get('message')
    if not user_input:
        return jsonify({'response': 'Please provide a message.'}), 400
    sentiment = sentiment_model(user_input)[0]
    response = generate_response(user_input, sentiment)
    return jsonify({'response': response})

def generate_response(user_input, sentiment):
    if sentiment['label'] == 'NEGATIVE':
        prompt = f"The user is upset, respond with empathy and support: {user_input}"
    else:
        prompt = f"Respond to the following query: {user_input}"
    input_ids = tokenizer.encode(prompt, truncation=True, padding=True, max_length=512, return_tensors="tf")
    output = model.generate(input_ids, max_length=250, num_beams=5, early_stopping=True)
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response


if __name__ == '__main__':
    app.run(debug=True, port=5000, host='0.0.0.0')


In [ ]:

import subprocess

# Stop any running Flask app
subprocess.run(['pkill', '-f', 'flask_chatbot_app.py'])

In [ ]:
!nohup python flask_chatbot_app.py &

In [ ]:
!sudo lsof -i -P -n | grep LISTEN

In [ ]:

import requests

# Define the URL of the Flask app
url = 'http://127.0.0.1:5000/chatbot'

# Send a request to the Flask app
response = requests.post(url, json={'message': 'Hello, how are you?'})
print(response.json())


In [ ]:
response = requests.post(url, json={'message': 'I was wondering how can I go to Eiffel Tower from the airport using the train and subway?'})
print(response.json())